# Lesson 1: Agentic RAG - Code Along 4 - Agentic RAG

Now make the RAG agentic.
Before, we used a fixed pipeline.
Now we implement the search as a tool, which the LLM agent can call.
For this, we build upon the persistent RAG pipeline from before.

In [1]:
# dependencies
import json

from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase
from anthropic import Anthropic

In [2]:
# define search tool
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [3]:
# build definition for the search tool so the LLM knows about and how to use it
search_tool = {
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [4]:
# connect to db and load data
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [5]:
# initialize anthropic client for LLM API calls
anthropic_client = Anthropic()

In [6]:
# system prompt
INSTRUCTIONS = '''
Your task is to answer the course participants' questions about the course.

Retrieve context from the course's FAQ knowledge base using the search tool.
Use the context to find relevant information and provide accurate answers.
Answer only based on the information in the knowledge base.
Do not answer if the question is not already answered in the knowledge base.

You may optimize the search terms in order to increase the chance of finding
relevant results.
If the search does not yield any results, you may repeat the search up to two
times with optimized queries and related terms.

If also repeated search does not yield any relevant information and the answer
is not found in the context, respond with "I don't know."
Do not make up things!
'''

# initialize message history - allows appending newer ones
messages = [
    {
        "role": "user",
        "content": "I just discovered the course. Can I join it?"
    }
]

In [7]:
# query the LLM on the question and provide tool
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages,
    tools=[search_tool],
)

In [8]:
# print full response
for item in response:
    print(item)

('id', 'msg_01W2sqMZC7baHMAMEvFhxqoZ')
('container', None)
('content', [ToolUseBlock(id='toolu_01QT7VE1tFPgGyqGqt7p7bwg', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enrollment registration'}, name='search', type='tool_use')])
('model', 'claude-haiku-4-5-20251001')
('role', 'assistant')
('stop_details', None)
('stop_reason', 'tool_use')
('stop_sequence', None)
('type', 'message')
('usage', Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=748, output_tokens=58, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [9]:
# see that currently there is only one message
for message in messages:
    print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]


In [10]:
# append the model's resonse to the message history to enable agentic behavrior
messages.append({"role": "assistant", "content": response.content})
print("Appended tool use to message history.")

Appended tool use to message history.


In [11]:
# print message history to see tool use was added
for message in messages:
    print(message)

{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
{'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01QT7VE1tFPgGyqGqt7p7bwg', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enrollment registration'}, name='search', type='tool_use')]}


In [12]:
# call the tool

# initialize empty list to contain tool results
tool_results = []

# iterate through all tool call and get results
# here, this is actually not strictly necessary, because there is just one call
# but in more complex settings the agent could also have made two or more calls
# in cases like that, this loop will get a response for each one
for block in response.content:
    if block.type == "tool_use" and block.name == "search":
        results = search(**block.input)
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": json.dumps(results),
        })

# check what this produced
# print dictionary in JSON format for better readability
for result in tool_results:
    print(json.dumps(result, indent=2))

{
  "type": "tool_result",
  "tool_use_id": "toolu_01QT7VE1tFPgGyqGqt7p7bwg",
  "content": "[{\"id\": \"74eb249bbf\", \"course\": \"llm-zoomcamp\", \"section\": \"General Course-Related Questions\", \"question\": \"I just discovered the course. Can I still join?\", \"answer\": \"Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.\"}, {\"id\": \"977bf7786c\", \"course\": \"llm-zoomcamp\", \"section\": \"General Course-Related Questions\", \"question\": \"Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\", \"answer\": \"You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\"}, {\"id\": \"bd31146b0e\", \"course\": \"llm-zoomcamp\", \"section\": \"General Course-Related Quest

In [13]:
# add the result to the message history
messages.append({"role": "user", "content": tool_results})

# check new history
for message in messages:
    print(message)

{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
{'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01QT7VE1tFPgGyqGqt7p7bwg', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enrollment registration'}, name='search', type='tool_use')]}
{'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01QT7VE1tFPgGyqGqt7p7bwg', 'content': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}, {"id": "977bf7786c", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning

In [14]:
# query the LLM again with tool/search result in messages
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages,
    tools=[search_tool],
)

In [15]:
# print final response
print(response.content[0].text)

Great question! **Yes, you can join the course!** 

According to the course FAQ, you can start learning and submitting homework even if you just discovered it. Here are the key points:

- **You can join anytime** - you don't need to wait for formal confirmation. You can simply start learning and submitting homework while the submission form is open.
- **No official registration needed** - Registration is optional and just used to gauge interest. It's not checked against any registered list, so you can begin without it.

However, there's one important thing to note: **If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.** This is because certificates are only awarded during live cohort runs when peer-review happens.

To get started, I recommend checking out:
- The [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)
- The [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)
- T

### Experiment with History

I want to see what happens when we send only parts of the history.
Like for example I want to try omitting the user question and see if the API
even accepts this, and if the model is confused then because it lacks context.

In [16]:
# exclude the user question and start with tool call right away
messages[1:3]

[{'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01QT7VE1tFPgGyqGqt7p7bwg', caller=DirectCaller(type='direct'), input={'query': 'how to join the course enrollment registration'}, name='search', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01QT7VE1tFPgGyqGqt7p7bwg',
    'content': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}, {"id": "977bf7786c", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without reg

The assistant's search query is still rather conclusive about the user's intend.
My guess is that it will still answer to the question.
I assume that it will even provide a rather nice answer, because there's also
still the system prompt.
But let's see if the API reguses to accept this.

In [17]:
# query on manipulated message history and print result right away
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages[1:3],
    tools=[search_tool],
)
print(response.content[0].text)

Hello! I'm here to help answer your questions about the course. I can see there are several topics covered in the course FAQ, including:

- **General enrollment and registration** - You can join the course at any time, and don't need a confirmation email to get started
- **Certificates** - Information about how to earn a certificate
- **Course structure** - How to follow the weekly workflow and get started
- **Course scheduling** - When the next course will be offered

Please feel free to ask me any specific questions you have about the course, and I'll search the knowledge base to find the answers for you!


Interesting! Very similar to what I guessed, but still a little different.
The model still provided the information answering the question, but it did not
assume the user asked about this particular thing.
This was Haiku.
Let's see how Sonnet and Opus behave.
Perhaps they will be able to guess the user's question.
If they don't then this seems to be a general pattern of behavior.

In [18]:
# try sonnet 4.5, which is the second to latest release
response = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages[1:3],
    tools=[search_tool],
)
print(response.content[0].text)

Hello! How can I help you with the course today? I have access to the course FAQ and can answer questions about enrollment, course materials, certificates, homework, and other course-related topics.


In [19]:
# now try the latest opus, 4.8
# this is the strongest model except for fable and mythos, which are banned :D
response = anthropic_client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    system=INSTRUCTIONS,
    messages=messages[1:3],
    tools=[search_tool],
)
print(response.content[0].text)

Hello! I'd be happy to help you with questions about the course. Could you let me know what specifically you'd like to know?

In the meantime, here's some helpful information I found:

- **Can you still join?** Yes! Even if you just discovered the course, you can still join. If you want a certificate, just make sure to submit your project while submissions are still being accepted.

- **Do you need to register?** No registration is strictly required. You're accepted automatically, and you can start learning and submitting homework right away (while the form is open). Registration just helps gauge interest before the start date.

Feel free to ask me anything specific about the course!


Okay they all behave similarly.
I assume this (also) is because of the use system prompt.
Let's just omit the system prompt and see if that works.

In [21]:
# query on manipulated message history and print result right away
response = anthropic_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    #system=INSTRUCTIONS,
    messages=messages[1:3],
    tools=[search_tool],
)
print(response.content[0].text)

Hello! I found some information about joining the course. Here's what you should know:

**Can I still join?**
Yes, you can still join the course anytime! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

**Do I need to register?**
Registration is optional. You don't need a confirmation email to start learning. You can begin immediately and submit homework without officially registering - registration is mainly just to gauge interest before the course starts.

**How do I get started?**
1. Check out the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)
2. Visit the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) to see deadlines

**Typical workflow:**
1. Watch the lesson videos
2. Work through lesson notebooks/code
3. Read homework instructions on GitHub
4. Submit answers through the course pl

Yes, removing the system prompt actually removed this particular behavior.
It specifically mentioned the user's questions.
Without it, it just summarizes the information and later asks if there was
anything specific the user wanted to know.

Just for fun, now let's send **only** the tool results and nothing else.
Not even the assistant's own decision to make that call in the first place.

In [34]:
# slicing like this gets us the tool results only
messages[2:3]

[{'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01QT7VE1tFPgGyqGqt7p7bwg',
    'content': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}, {"id": "977bf7786c", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {"id": "bd31146b0e", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question

In [35]:
# send ONLY the tool results
try:
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        #system=INSTRUCTIONS,
        messages=messages[2:3],
        tools=[search_tool],
    )
    print(response.content[0].text)
except Exception as e:
    print(e)

Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.content.0: unexpected `tool_use_id` found in `tool_result` blocks: toolu_01QT7VE1tFPgGyqGqt7p7bwg. Each `tool_result` block must have a corresponding `tool_use` block in the previous message.'}, 'request_id': 'req_011CcGTHfNZfPoGjsR34KW9p'}


The API doesn't let us do this.
Every tool result block needs to have a tool call block before it.

## The Agentic Loop

So far, we kind of still used tool calling by hand.
Now, we will implement the agentic loop enabling multiple consecutive tool calls
without human intervention.

In [65]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
# define helper function for function calls
# right now only contains search tool, but can be extended
def make_call(block):
    
    # if block name is "search", perform the search with the input arguments
    if block.name == "search":
        result = search(**block.input)

    # return search results in expected format
    return {
        "type": "tool_result",
        "tool_use_id": block.id,
        "content": json.dumps(result, indent=2),
    }

In [45]:
# user question
question = "What is this course about?"

# format user question for sending to anthropic
messages = [{"role": "user", "content": question}]

# count iterations of the loop (starts with 1 here)
it = 1

while True:
    
    # print number of iteration
    print(f"\nIteration #{it}...")

    # get response from model using new system prompt and user question
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        system=instructions,
        messages=messages,
        tools=[search_tool],
    )
    
    # append assistant message to history 
    # most likely tool call, because it's pushed to do it
    messages.append({"role": "assistant", "content": response.content})
    
    # initialize empty list for containing tool results
    tool_results = []
    
    # iterate through block in response
    # print all blocks in nice format
    # if it's a tool, get the result and add it to toll results list
    for block in response.content:
        if block.type == "tool_use":
            print("tool_use:", block.name, block.input)
            tool_results.append(make_call(block))
        elif block.type == "text":
            print("ASSISTANT:")
            print(block.text)
    
    # if there are any tool results, add them to the message history
    if tool_results:
        messages.append({"role": "user", "content": tool_results})
    
    # count iterations
    it += 1
    
    # check break condition
    # stops if either of two conditions evaluate to true
    # either the agent doesn't want to call any further tools
    # obviously there need to be no further iterations for tool calling then
    # or if it already made 5 iterations, which prevents token burning
    if response.stop_reason != "tool_use" or it == 5:
        break



Iteration #1...
ASSISTANT:
I'll search the FAQ for information about what this course is about.
tool_use: search {'query': 'course about overview description'}
tool_use: search {'query': 'course objectives goals topics'}

Iteration #2...
ASSISTANT:
Great! Based on the search results, I can see this is the **LLM Zoomcamp** course. Let me search for more specific information about what the course actually covers.
tool_use: search {'query': 'LLM Zoomcamp modules content topics'}
tool_use: search {'query': 'large language models RAG'}

Iteration #3...
ASSISTANT:
Perfect! Now I have a clearer picture. Here's what the **LLM Zoomcamp** course is about:

## Course Overview

**LLM Zoomcamp** is a practical course focused on building applications with Large Language Models (LLMs). Key topics include:

### Main Topics:
1. **Module 1: Agentic RAG** - Building intelligent systems where LLMs can dynamically call tools and functions for tasks like document search, database queries, API calls, and ca

In [ ]:
# this actually keeps the full message history
# this is cool because it can be extended even further
messages

[{'role': 'user', 'content': 'What is this course about?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll search the FAQ for information about what this course is about.", type='text'),
   ToolUseBlock(id='toolu_01LFZhxULXk76nVtx8BkFrAf', caller=DirectCaller(type='direct'), input={'query': 'course about overview description'}, name='search', type='tool_use'),
   ToolUseBlock(id='toolu_01Uv3keEbA9Psytynk4aYrio', caller=DirectCaller(type='direct'), input={'query': 'course objectives goals topics'}, name='search', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01LFZhxULXk76nVtx8BkFrAf',
    'content': '[\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related 

Now add the tool call loop in a callable function.

In [63]:
# this doesn't work: str.concat(instructions, "hi")
# concatenate two strings RIGHT NOW
test_max_iter = 5
instructions + " The max number of iterations is " + str(test_max_iter)


"You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore. The max number of iterations is 5"

In [80]:
def agent_loop(instructions, question, model="claude-haiku-4-5", max_iter=5) -> str:

    # add information about max number of iterations to system prompt
    instructions += (
        f"\n\nThe max number of iterations is {max_iter}. "
        "One iteration is one response or tool call. "
        "Do not make more than {max_iter} iterations. "
        " You will be warned before the last iteration, so you can make a "
        "final answer based on what you know so far then."
    )

    # format user question for sending to anthropic
    messages = [{"role": "user", "content": question}]

    # count iterations of the loop (starts with 1 here)
    it = 1

    while True:
        
        # print number of iteration
        print(f"\nIteration #{it}...")

        # get response from model using new system prompt and user question
        response = anthropic_client.messages.create(
            model=model,
            max_tokens=1024,
            system=instructions,
            messages=messages,
            tools=[search_tool],
        )
        
        # append assistant message to history 
        # most likely tool call, because it's pushed to do it
        messages.append({"role": "assistant", "content": response.content})
        
        # initialize empty list for containing tool results
        tool_results = []
        
        # iterate through block in response
        # print all blocks in nice format
        # if it's a tool, get the result and add it to toll results list
        for block in response.content:
            if block.type == "tool_use":
                print("tool_use:", block.name, block.input)
                tool_results.append(make_call(block))
            elif block.type == "text":
                latest_answer = block.text
                print("ASSISTANT:")
                print(latest_answer)
        
        # warn agent about last iteration before loop break
        # this is appended at iteration 4
        # next, it counter is increased to 5, and one more answer can be made
        # then it counter will be set to 6, which is a hard break
        # the agent should also make no further tool calls anyway then 
        if it == (max_iter - 1):
            tool_results.append({
                "type": "text",
                "text": (
                    "You now used up all tool calls! "
                    "Do not make any further tool calls and produce a final answer "
                    "based on what you know so far. "
                    "If the tool calls did not yield results and you cannot base "
                    "an answer on them, tell the user that you don't know. "
                    "Also, be transparent about this limitation to the user! "
                    "They must know why you could not go on!"
                )
            })
        
        # if there are any tool results, add them to the message history
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
        
        # count iterations
        it += 1
        
        # check break condition
        # stops if either of two conditions evaluate to true
        # either the agent doesn't want to call any further tools
        # obviously there need to be no further iterations for tool calling then
        # or if it already made 5 iterations, which prevents token burning
        if response.stop_reason != "tool_use" or it == (max_iter + 1):
            break
        
    return latest_answer

In [67]:
# try a question with a typo
# right now we use dictionary search, so it won't find "Olama"
# it should make at least 2 searches
agent_loop(instructions, "How do I run Olama locally?")


Iteration #1...
ASSISTANT:
I'll search for information about running Ollama locally.
tool_use: search {'query': 'run Ollama locally'}

Iteration #2...
ASSISTANT:
Great! I found detailed information about running Ollama locally. Here's how to do it:

## How to Run Ollama Locally

### 1. **Install Ollama**

Visit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:

- **macOS**: Download the `.pkg` and install it
- **Windows**: Download the `.msi` and install it
- **Linux**: Run this command in the terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

### 2. **Start Ollama with a Model**

Once installed, open a terminal and run:

```bash
ollama run llama3
```

This will:
- Download the LLaMA 3 model (~4GB)
- Start the model locally
- Open a chat-like interface where you can type questions

### 3. **Verify Ollama is Running**

Test the local Ollama server with:

```bash
curl http://localhost:11434
```

You should get a response

'Great! I found detailed information about running Ollama locally. Here\'s how to do it:\n\n## How to Run Ollama Locally\n\n### 1. **Install Ollama**\n\nVisit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:\n\n- **macOS**: Download the `.pkg` and install it\n- **Windows**: Download the `.msi` and install it\n- **Linux**: Run this command in the terminal:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n### 2. **Start Ollama with a Model**\n\nOnce installed, open a terminal and run:\n\n```bash\nollama run llama3\n```\n\nThis will:\n- Download the LLaMA 3 model (~4GB)\n- Start the model locally\n- Open a chat-like interface where you can type questions\n\n### 3. **Verify Ollama is Running**\n\nTest the local Ollama server with:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n```json\n{"models": [...]}\n```\n\n### 4. **Use Ollama with Python**\n\nInstall the Python client:\n\n```bash\npip in

Ah lol it was too clever for this. Let me try again.

I will have to frame the question in a way that makes sure it will search for
"Olama" first, regardless of it knowing it's actually called "Ollama", because
I want to see it perform multiple searches.

In [70]:
# try a question with a typo
# right now we use dictionary search, so it won't find "Olama"
# it should make at least 2 searches
response = agent_loop(
    instructions=instructions,
    question=(
        "How do I run Olama locally? "
        "Find out using this original question first before modifying it."
        "I think the program is really called Olama, not Ollama, but maybe I'm "
        "wrong. "
        "But try to look for Olama first. "
        "The instructor always calls it like that, so maybe it's some custom "
        "stuff he uses."
    )
)


Iteration #1...
ASSISTANT:
I'll search for information about Olama first, as you mentioned, and then we can explore further if needed.
tool_use: search {'query': 'Olama run locally'}

Iteration #2...
ASSISTANT:
Great! I found information about **Ollama** in the course FAQ. It looks like the program is actually called **Ollama** (not Olama), but the course material has detailed instructions on how to run it locally. Let me search for more specific information about running Ollama locally.
tool_use: search {'query': 'Ollama install locally setup'}

Iteration #3...
ASSISTANT:
Perfect! I have found comprehensive information about running Ollama locally. Here's what I found:

## How to Run Ollama Locally

**Note:** The program is actually called **Ollama** (not Olama), though your instructor may refer to it in different ways!

### Installation Steps:

1. **Download and Install Ollama** from [https://ollama.com/download](https://ollama.com/download) for your operating system:
   - **macOS**

I cannot make it hit the limit I built in, so I now lower the limit.

In [81]:
# I try to push it to the max number of iterations now
# as far as I know, this answer is not in the FAQ
response = agent_loop(
    instructions=instructions,
    question=(
        "Who is the instructor of this course? "
        "Search until you find the answer and do not stop!"
    ),
    max_iter=2
)


Iteration #1...
ASSISTANT:
I'll search for information about the course instructor.
tool_use: search {'query': 'course instructor'}
tool_use: search {'query': 'who is the instructor'}
tool_use: search {'query': 'instructor name professor'}

Iteration #2...
ASSISTANT:
I apologize, but I was unable to find the instructor's name in the course FAQ database. 

Based on my searches using keywords like "course instructor," "who is the instructor," and "instructor name professor," the FAQ results only returned information about general course logistics, modules, and technical troubleshooting—but no entries specifically identifying the course instructor.

**Why I couldn't continue:** I've exhausted my allocation of search queries (I can only make 2 iterations/rounds of searches), and the searches I performed didn't yield the instructor information you're looking for.

**What I did find:** The course appears to be the **LLM Zoomcamp** offered by **DataTalks Club**, based on the documentation an

Cool! This works!
I actually told it to be transparent about the limit, so I can see what happens.

### Restricting Off-Topic Questions

So far, the assistant will answer to anything not already restricted by
Anthropic's usage policy.
But it's not limited to answer questions about this course.
It will also try finding out who's Rononoa Zoro if I ask it.
It will now be limited to answer questions about the course only.

In [83]:
# see if it will try finding out who's Roronoa Zoro
# this is not course related
response = agent_loop(
    instructions=instructions,
    question=("Who is Roronoa Zoro?")
)


Iteration #1...
ASSISTANT:
I appreciate the question, but Roronoa Zoro is actually a fictional character from the anime and manga series "One Piece," not related to course material. 

Since I'm a course teaching assistant, I'm equipped to help you with questions related to the course content. Let me check if there's anything in the course FAQ that might be relevant:
tool_use: search {'query': 'Roronoa Zoro'}

Iteration #2...
ASSISTANT:
As expected, there's no course-related information about this character in our FAQ database.

Is there something related to the course material that I can help you with instead? Feel free to ask any questions about the course topics, assignments, policies, or other academic matters!


Wow interesting!
Anthropic already did a pretty good job at this!
Maybe the system prompt also helps.
In any case, the model cannot be fooled as easily.

I'll try the example from the course material.

In [84]:
# see if it will try finding out who's Roronoa Zoro
# this is not course related
response = agent_loop(
    instructions=instructions,
    question=("What's queen's gambit?")
)


Iteration #1...
ASSISTANT:
I'll search the course FAQ for information about Queen's Gambit.
tool_use: search {'query': "queen's gambit"}

Iteration #2...
ASSISTANT:
The search didn't return results about Queen's Gambit as a chess opening. Let me try a different search to see if this is mentioned in the course materials.
tool_use: search {'query': 'chess gambit opening'}

Iteration #3...
ASSISTANT:
Based on my search of the course FAQ database, I'm not finding any information about "Queen's Gambit" in the course materials. 

This course appears to be **llm-zoomcamp**, which focuses on LLM (Large Language Model) concepts, RAG (Retrieval-Augmented Generation), vector search, evaluation, and related topics. It doesn't seem to cover chess openings or the Queen's Gambit.

**Queen's Gambit**, if you're referring to the chess opening, is a classic chess move where White (playing first) sacrifices a pawn with 1.d4 d5 2.c4 to gain central control and development advantage.

However, if your que

Okay so it wasn't sure and it at least tried.
So, some cases are covered well already, and I think Zoro was too obvious.
But now let's still add these guardrails.
We will just tell it in the system prompt.

In [86]:
# exchange the system prompt for a more restricted one
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

response = agent_loop(instructions, "what's queen gambit?")


Iteration #1...
ASSISTANT:
I'll search the course FAQ to see if there's any information about Queen Gambit in relation to the course.
tool_use: search {'query': 'queen gambit'}

Iteration #2...
ASSISTANT:
The search returned no results. This suggests that "Queen Gambit" is not related to the course material or logistics. 

Since I'm a course teaching assistant and can only answer questions about the course or its logistics using the course FAQ, I'm not able to help with this question. Queen Gambit appears to be an off-topic question (it's a chess opening or a TV series, neither of which is covered in the course FAQ).

Is there anything related to the course itself that I can help you with?
